In [24]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager

import pandas as pd
import numpy as np
import re

In [25]:
options = Options()

#options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-infobars')
options.add_argument('--disable-extensions')
options.add_argument('--disable-notifications')
options.add_argument('--window-size=1724,968')
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.110 Safari/537.36"
options.add_argument(f'user-agent={user_agent}')
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [26]:
# load product href csvs
red_href_df = pd.read_csv("data/Red_wine_hrefs.csv")
white_href_df = pd.read_csv("data/White_wine_hrefs.csv")
sparkling_href_df = pd.read_csv("data/Champagne-Sparkling_wine_hrefs.csv")

# Combine the DataFrames into a single DataFrame
href_df = pd.concat([red_href_df, white_href_df, sparkling_href_df], ignore_index=True)

In [27]:
href_df

,href,type
0,https://www.danmurphys.com.au/product/DM_17275...,Red
1,https://www.danmurphys.com.au/product/DM_10535...,Red
2,https://www.danmurphys.com.au/product/DM_90074...,Red
3,https://www.danmurphys.com.au/product/DM_90104...,Red
4,https://www.danmurphys.com.au/product/DM_14446...,Red
...,...,...
6542,https://www.danmurphys.com.au/product/DM_13411...,Champagne-Sparkling
6543,https://www.danmurphys.com.au/product/DM_12199...,Champagne-Sparkling
6544,https://www.danmurphys.com.au/product/DM_10851...,Champagne-Sparkling
6545,https://www.danmurphys.com.au/product/DM_10750...,Champagne-Sparkling


# Going to Scrape:
### username
### rating
### date

In [45]:
%%time

#i_failure = 475 # i of last failure

# used for development
no_product_href = "https://www.danmurphys.com.au/product/DM_855655/riddoch-coonawarra-shiraz-187ml?isFromSearch=false&isPersonalised=false&pageName=category_page"
multi_review_page_href = "https://www.danmurphys.com.au/product/DM_119799/pepperjack-malbec?isFromSearch=false&isPersonalised=false&pageName=member_offers"
no_review_href = "https://www.danmurphys.com.au/product/DM_968821/catalina-sounds-pinot-noir?isFromSearch=false&isPersonalised=false&pageName=category_page"

hrefs_subset= []
reviews = []

total_wines = len(href_df)
starting_i = 0
for i in range(starting_i, len(href_df), 5):

    
    # get href
    href = href_df.iloc[i]["href"]
    hrefs_subset.append(href)
    driver.get(multi_review_page_href)


    product_exists = False
    try:
        review_cards_block = driver.find_element(By.CLASS_NAME, "not-found__content")
        print("No product page for i =", i, "-- href +", href)
    except NoSuchElementException:
        product_exists = True

    # check if product page exists
    if product_exists:
        is_reviewed = False
        try:
            review_cards_block = driver.find_element(By.CLASS_NAME, "customer-reviews__cards")
            is_reviewed = True
        except NoSuchElementException:
            print("!!! no product reviews for i =", i, "-- href =", href)
    
        # check if product has reviews
        if is_reviewed:
            review_cards = review_cards_block.find_elements(By.CLASS_NAME, "customer-reviews__card")
            
            user = review_cards_block.find_elements(By.CLASS_NAME, "review-card__sdesc-link")

            while True:
                try:

                    button = driver.find_element(By.XPATH, "//button[@aria-label='Next page button']")
                    
                    if button.is_enabled():
                        # Click the button
                        button.click()
                        print("Button clicked!")
                    else:
                        print("Next button is disabled. No more review pages.")
                        break

                
                except NoSuchElementException:
                    print("Couldn't find next button. No more review pages")
                    break





        
        else:
            reviews.append([])
        
    else:
        reviews.append([])

    
    break
        

SCRAPING...
Button clicked!
Button clicked!
Next button is disabled. No more review pages.
CPU times: user 49.6 ms, sys: 0 ns, total: 49.6 ms
Wall time: 29 s


In [36]:
len(review_cards)

4

In [35]:
reviews

[[]]

In [38]:
href

'https://www.danmurphys.com.au/product/DM_172754/penfolds-bin-128-shiraz-2019?isFromSearch=false&isSponsored=true&pageName=category_page&promoteIQCampaignID=277455'

In [ ]:
            """while True:
                try:
                    # Wait for the button to be clickable
                    button = WebDriverWait(driver, 5).until(
                        EC.element_to_be_clickable((By.XPATH, "//button[@aria-label='Next page button']"))
                    )

                    # Click the button
                    button.click()"""